# Evaluation and Interpretation

This notebook evaluates the three predictive models developed and frozen in `03-model-development.ipynb` using the held-out NHANES test cohort.

The test cohort was separated from the training cohort before metabolic phenotype discovery and was not used for model fitting or hyperparameter selection. The fitted training-set scaler and K-means model were used previously to assign test participants to the learned metabolic phenotypes.

Three models are evaluated:

1. Logistic regression baseline
2. XGBoost without metabolic phenotype
3. XGBoost with metabolic phenotype

The frozen models are applied to the held-out test cohort once. Their predictions are then used to compare overall predictive performance, assess whether adding metabolic phenotype improves prediction, examine performance within each metabolic phenotype, and interpret XGBoost predictions using SHAP.

## Loading the held-out test cohort and fitted models

The held-out test cohort from `02-metabolic-phenotyping.ipynb` is loaded from `test_clustered.csv`.

The three fitted model pipelines saved in `03-model-development.ipynb` are loaded from `data/models`. These pipelines already contain the preprocessing transformations and model parameters learned exclusively from the training cohort.

No preprocessing or model fitting is repeated in this notebook.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = DATA_DIR / "models"
RESULTS_DIR = Path("../results")

TEST_DATA_PATH = PROCESSED_DIR / "test_clustered.csv"

test_df = pd.read_csv(TEST_DATA_PATH)

logistic_model = joblib.load(
    MODEL_DIR / "logistic_regression.joblib"
)

xgb_base_model = joblib.load(
    MODEL_DIR / "xgboost_without_phenotype.joblib"
)

xgb_phenotype_model = joblib.load(
    MODEL_DIR / "xgboost_with_phenotype.joblib"
)

print(f"Test participants: {len(test_df):,}")
print("\nOutcome distribution:")
print(test_df["hepatic_steatosis"].value_counts().sort_index())

print("\nMetabolic phenotypes:")
print(test_df["metabolic_phenotype"].value_counts())

Test participants: 654

Outcome distribution:
hepatic_steatosis
0    278
1    376
Name: count, dtype: int64

Metabolic phenotypes:
metabolic_phenotype
Lower metabolic-risk     357
Higher metabolic-risk    297
Name: count, dtype: int64


## Preparing the held-out test predictors

The same predictor sets defined during model development are reconstructed from the held-out test cohort.

The base predictor set contains the 11 demographic, anthropometric, clinical and metabolic variables used by logistic regression and XGBoost without phenotype information. The phenotype-enhanced predictor set additionally contains `metabolic_phenotype`.

`SEQN`, CAP, the numerical cluster label and the hepatic steatosis outcome are not included as predictors.

The true hepatic steatosis outcome is retained separately as `y_test` so that the model predictions can subsequently be compared with the known CAP-derived outcome.

In [2]:
numeric_features = [
    "age",
    "bmi",
    "waist_cm",
    "systolic_bp",
    "diastolic_bp",
    "hdl_mg_dl",
    "triglycerides_mg_dl",
    "glucose_mg_dl",
]

categorical_features = [
    "sex",
    "race_ethnicity",
    "diabetes_status",
]

base_features = numeric_features + categorical_features
features_with_phenotype = base_features + ["metabolic_phenotype"]

X_test_base = test_df[base_features].copy()
X_test_with_phenotype = test_df[features_with_phenotype].copy()

y_test = test_df["hepatic_steatosis"].copy()

print(f"X_test_base:            {X_test_base.shape}")
print(f"X_test_with_phenotype:  {X_test_with_phenotype.shape}")
print(f"y_test:                 {y_test.shape}")

X_test_base:            (654, 11)
X_test_with_phenotype:  (654, 12)
y_test:                 (654,)


## Generating held-out test predictions

The three frozen model pipelines are now applied to the held-out test cohort.

Each model generates a predicted probability of hepatic steatosis for every test participant. A probability threshold of `0.50` is also used to generate binary classifications, where `1` represents predicted hepatic steatosis and `0` represents predicted absence of hepatic steatosis.

The models are not refitted or modified using the test data. These predictions therefore provide the basis for the final held-out evaluation of model performance.

In [3]:
# Generate probabilities from the three frozen models
logistic_probability = logistic_model.predict_proba(
    X_test_base
)[:, 1]

xgb_base_probability = xgb_base_model.predict_proba(
    X_test_base
)[:, 1]

xgb_phenotype_probability = xgb_phenotype_model.predict_proba(
    X_test_with_phenotype
)[:, 1]

# Convert probabilities to binary predictions
threshold = 0.50

logistic_prediction = (
    logistic_probability >= threshold
).astype(int)

xgb_base_prediction = (
    xgb_base_probability >= threshold
).astype(int)

xgb_phenotype_prediction = (
    xgb_phenotype_probability >= threshold
).astype(int)

print(f"Predictions generated: {len(y_test):,}")

print("\nPredicted steatosis cases:")
print(f"Logistic regression:       {logistic_prediction.sum():,}")
print(f"XGBoost:                   {xgb_base_prediction.sum():,}")
print(f"XGBoost + phenotype:       {xgb_phenotype_prediction.sum():,}")

Predictions generated: 654

Predicted steatosis cases:
Logistic regression:       401
XGBoost:                   414
XGBoost + phenotype:       408
